# LLM Adaptation Workflow
### Notebook 05 — Preference Alignment with DPO

---

## Why alignment matters

Fine-tuning teaches a model **what to say** — it learns facts, style, and domain knowledge. But it doesn't teach the model **how to say it well**.

After fine-tuning, a model might:
- Give technically correct but overly verbose answers
- Hedge excessively ("it depends", "it's hard to say")
- Respond in an inappropriate tone
- Occasionally produce harmful or misleading content

**Alignment** is the process of teaching the model to produce outputs that humans actually *prefer* — not just outputs that are technically correct.

---

## The RLHF pipeline

The classic approach is **RLHF (Reinforcement Learning from Human Feedback)**:

```
1. Collect preference data    ← Humans rank model outputs A vs B
2. Train a reward model       ← Learns to predict which outputs humans prefer
3. Use RL (PPO) to train LLM  ← Optimise model outputs to maximise reward
```

This is how GPT-4, Claude, and Gemini are aligned. The challenge: it requires running four models simultaneously (LLM, reference LLM, reward model, value model) and is notoriously unstable.

---

## DPO — a simpler alternative

**DPO (Direct Preference Optimisation)** achieves the same goal in a simpler way.

Instead of training a separate reward model and using RL, DPO reformulates the problem as a **classification** task:
- Given a prompt, a chosen response, and a rejected response
- Train the model to increase the probability of the chosen response
- And decrease the probability of the rejected response
- With a KL penalty to prevent the model from drifting too far from its original behaviour

DPO requires no RL, no reward model, and trains in a single pass — making it practical on consumer hardware.

---

## The data format

DPO training requires **preference pairs**: for each prompt, a preferred (chosen) response and a less preferred (rejected) response.

```json
{
  "prompt": "What was Apple's total revenue in 2023?",
  "chosen": "Apple reported total net sales of $383.3 billion for fiscal 2023, down 2.8% from $394.3 billion in 2022. The decline was primarily driven by weaker iPhone demand in China.",
  "rejected": "Apple made a lot of money in 2023. Revenue was very high and the company performed well overall."
}
```

In production, preference pairs come from:
- Human raters comparing model outputs
- A stronger model (e.g. GPT-4) ranking outputs from a weaker model
- Automated quality signals (factual accuracy checks, format compliance)

In [ ]:
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, PeftModel
from trl import DPOTrainer, DPOConfig

MODEL_DIR = Path("../models")

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

---

## 1. Building a preference dataset

A preference dataset consists of triplets: (prompt, chosen, rejected).

We construct a realistic financial services dataset that captures the kind of output quality differences that matter in practice:
- **Chosen**: specific, grounded in data, appropriately concise
- **Rejected**: vague, hedging, lacks specifics, or inappropriate tone

In [ ]:
# Preference dataset: each entry has a prompt, a good response (chosen), and a poor response (rejected)
# These represent the kinds of quality distinctions that matter in financial analysis

preference_data = [
    {
        "prompt": "What was Apple's total revenue for fiscal year 2023?",
        "chosen": "Apple reported total net sales of $383.3 billion for fiscal year 2023, a decrease of 2.8% from $394.3 billion in fiscal 2022. The decline was primarily driven by lower iPhone demand, partially offset by growth in Services revenue which rose 16% to $85.2 billion.",
        "rejected": "Apple had a lot of revenue in 2023. It was a big company and made lots of money. The exact figure might have changed so I'd recommend checking the latest reports."
    },
    {
        "prompt": "Summarise the key risks Microsoft identifies in its 10-K filing.",
        "chosen": "Microsoft's 10-K identifies five primary risk categories: (1) competitive threats from firms like Google, Amazon, and Apple in cloud and productivity; (2) cybersecurity and data privacy risks given the volume of enterprise data processed; (3) regulatory and antitrust risk across its global operations; (4) dependence on continued Azure adoption for growth; and (5) macroeconomic exposure through enterprise IT spending cycles.",
        "rejected": "Microsoft has many risks. Competition is one risk. They also have technology risks and legal risks. There are many other risks as well that are described in the document."
    },
    {
        "prompt": "How should an analyst interpret a decline in gross margin for a technology company?",
        "chosen": "A gross margin decline in a technology company warrants investigation across three dimensions: (1) product mix shift — lower-margin hardware growing faster than software/services; (2) cost pressures — component price increases, supply chain disruptions, or higher cloud infrastructure costs; and (3) competitive pricing — discounting to defend market share. The interpretation depends on whether the decline is structural (mix shift to a lower-margin business) or cyclical (temporary cost pressures expected to normalise).",
        "rejected": "A decline in gross margin could mean different things. It might be bad or it could be okay depending on the context. You should look at why it happened. Sometimes margins go down and sometimes they go up. It's hard to say without more information."
    },
    {
        "prompt": "What does JPMorgan's net interest income tell us about their business model?",
        "chosen": "JPMorgan's net interest income (NII) — the spread between what it earns on loans/investments and what it pays on deposits — is a core profitability driver for its retail and commercial banking segments. Rising NII in a high-rate environment reflects the bank's asset-sensitive balance sheet: assets (loans) reprice upward faster than liabilities (deposits). This makes JPMorgan a beneficiary of rate increases but exposes it to NII compression if rates fall or deposit outflows accelerate.",
        "rejected": "Net interest income is money that JPMorgan makes from interest. They get interest from loans and pay interest on deposits. The difference is their profit. This is important for banks."
    },
    {
        "prompt": "Explain the difference between revenue growth and earnings growth.",
        "chosen": "Revenue growth measures the rate at which a company's top-line sales are increasing. Earnings growth (typically EPS growth) measures profit growth after all costs are deducted. The gap between the two reflects margin dynamics: a company growing revenue 15% but earnings only 5% is seeing margin compression — costs rising faster than sales. Conversely, earnings growing faster than revenue signals operating leverage or improving cost efficiency. For growth-stage companies, revenue growth is often prioritised; for mature companies, earnings growth matters more to investors.",
        "rejected": "Revenue is the total sales and earnings is the profit. Revenue growth means sales are going up and earnings growth means profits are going up. Both are important financial metrics that investors look at."
    },
    {
        "prompt": "What are the trade-offs between using RAG versus fine-tuning for a financial document assistant?",
        "chosen": "RAG is preferable when: (1) the knowledge base changes frequently (quarterly filings, live prices); (2) auditability matters — you need to show which document grounded each answer; or (3) setup speed is critical. Fine-tuning is preferable when: (1) consistent analytical style and tone are important; (2) the query types are predictable and repetitive; or (3) inference latency is constrained and retrieval overhead is unacceptable. In practice, production financial assistants often combine both: fine-tuning for style and domain reasoning, RAG for live document retrieval.",
        "rejected": "RAG and fine-tuning are both ways to improve language models. RAG retrieves documents and fine-tuning trains the model. Both have pros and cons. You should choose based on your needs."
    },
]

dataset = Dataset.from_list(preference_data)
print(f"Preference dataset: {len(dataset)} examples")
print()
print("Example entry:")
print(f"  Prompt  : {dataset[0]['prompt']}")
print(f"  Chosen  : {dataset[0]['chosen'][:100]}...")
print(f"  Rejected: {dataset[0]['rejected'][:100]}...")

---

## 2. Understanding what makes a response "chosen" vs "rejected"

Let's analyse our preference dataset to understand the quality dimensions we're teaching.

In [ ]:
import pandas as pd

df = dataset.to_pandas()
df["chosen_len"]   = df["chosen"].str.split().str.len()
df["rejected_len"] = df["rejected"].str.split().str.len()

print("Response length comparison (words):")
print(f"  Chosen   — mean: {df['chosen_len'].mean():.0f}, min: {df['chosen_len'].min()}, max: {df['chosen_len'].max()}")
print(f"  Rejected — mean: {df['rejected_len'].mean():.0f}, min: {df['rejected_len'].min()}, max: {df['rejected_len'].max()}")
print()
print("Quality dimensions our preference data teaches:")
print("  ✓ Specificity     — chosen responses include concrete numbers and facts")
print("  ✓ Structure       — chosen responses use enumeration, categories, comparisons")
print("  ✓ Depth           — chosen responses explain *why*, not just *what*")
print("  ✓ Appropriate tone — chosen responses match professional analyst register")
print("  ✗ Avoided: hedging, vagueness, lack of domain knowledge")

---

## 3. Reward model — what it does and how it's trained

Before DPO simplified the pipeline, alignment required training a **reward model** — a separate neural network that predicts a scalar quality score for any (prompt, response) pair.

This is still used in large-scale RLHF (as in the original InstructGPT paper). Let's build one to understand the concept.

In [ ]:
from transformers import AutoModelForSequenceClassification

# A reward model is typically a language model with a classification head
# It takes (prompt + response) as input and outputs a single number: the reward
# We use a small model for demonstration

REWARD_MODEL_ID = "distilbert-base-uncased"

reward_tokeniser = AutoTokenizer.from_pretrained(REWARD_MODEL_ID)
reward_model = AutoModelForSequenceClassification.from_pretrained(
    REWARD_MODEL_ID,
    num_labels=1,          # Single output: the scalar reward
)
reward_model = reward_model.to(device)

print(f"Reward model: {REWARD_MODEL_ID}")
print(f"Output: single scalar reward score")
print(f"Parameters: {sum(p.numel() for p in reward_model.parameters())/1e6:.0f}M")

In [ ]:
# Prepare training data for the reward model
# Each example: concatenate prompt + response, label = 1 (chosen) or 0 (rejected)

reward_examples = []
for ex in preference_data:
    reward_examples.append({
        "text": ex["prompt"] + " [SEP] " + ex["chosen"],
        "label": 1.0   # preferred
    })
    reward_examples.append({
        "text": ex["prompt"] + " [SEP] " + ex["rejected"],
        "label": 0.0   # not preferred
    })

reward_dataset = Dataset.from_list(reward_examples)

def tokenise_reward(examples):
    return reward_tokeniser(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

tokenised_reward = reward_dataset.map(tokenise_reward, batched=True)
tokenised_reward = tokenised_reward.rename_column("label", "labels")
tokenised_reward.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Reward model training set: {len(tokenised_reward)} examples ({len(preference_data)} chosen + {len(preference_data)} rejected)")

In [ ]:
from transformers import Trainer, TrainingArguments

reward_training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "reward_model"),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    logging_steps=5,
    save_strategy="no",
    use_mps_device=(device == "mps"),
    fp16=False,
    bf16=False,
    report_to="none",
)

reward_trainer = Trainer(
    model=reward_model,
    args=reward_training_args,
    train_dataset=tokenised_reward,
)

print("Training reward model...")
reward_trainer.train()
print("Reward model trained.")

In [ ]:
# Test the reward model: does it assign higher scores to chosen vs rejected responses?

def get_reward(prompt, response):
    text = prompt + " [SEP] " + response
    inputs = reward_tokeniser(
        text, return_tensors="pt", truncation=True, max_length=256
    ).to(device)
    with torch.no_grad():
        output = reward_model(**inputs)
    return output.logits.squeeze().item()

print("Reward model scoring (higher = more preferred):")
print()
for ex in preference_data[:3]:
    r_chosen   = get_reward(ex["prompt"], ex["chosen"])
    r_rejected = get_reward(ex["prompt"], ex["rejected"])
    correct = "✓" if r_chosen > r_rejected else "✗"
    print(f"{correct} Prompt: {ex['prompt'][:60]}...")
    print(f"    Chosen score  : {r_chosen:+.3f}")
    print(f"    Rejected score: {r_rejected:+.3f}")
    print()

---

## 4. DPO fine-tuning

Now we apply DPO to the fine-tuned model from Notebook 04. This directly optimises the model to prefer chosen responses over rejected ones — without needing the reward model or RL.

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = MODEL_DIR / "tinyllama-financial-lora" / "final_adapter"

dpo_tokeniser = AutoTokenizer.from_pretrained(MODEL_ID)
if dpo_tokeniser.pad_token is None:
    dpo_tokeniser.pad_token = dpo_tokeniser.eos_token

# Load the fine-tuned model from Notebook 04 as our starting point
# (If notebook 04 hasn't been run, load the base model instead)
if adapter_path.exists():
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
    dpo_model = PeftModel.from_pretrained(base, str(adapter_path))
    dpo_model = dpo_model.merge_and_unload()  # Merge adapter into base for DPO training
    print("Loaded fine-tuned model from Notebook 04.")
else:
    dpo_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
    print("Loaded base model (run Notebook 04 first for the fine-tuned version).")

dpo_model = dpo_model.to(device)

In [ ]:
# DPO also uses LoRA adapters — we don't want to full-fine-tune the model again
dpo_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                   # Smaller rank for alignment stage
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

dpo_config = DPOConfig(
    output_dir=str(MODEL_DIR / "tinyllama-dpo"),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    beta=0.1,              # KL penalty — higher = model stays closer to original. Typical range: 0.01–0.5
    max_length=512,
    max_prompt_length=256,
    use_mps_device=(device == "mps"),
    fp16=False,
    bf16=False,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_config,
    train_dataset=dataset,
    tokenizer=dpo_tokeniser,
    peft_config=dpo_lora_config,
)

print("DPO trainer configured.")
print(f"beta = {dpo_config.beta}  (KL divergence penalty — controls how far the model drifts)")

In [ ]:
print("Running DPO training...")
dpo_trainer.train()

# Save the DPO-aligned model
dpo_trainer.save_model(str(MODEL_DIR / "tinyllama-dpo" / "final"))
print("DPO training complete. Aligned model saved.")

---

## 5. Comparing base, fine-tuned, and aligned responses

In [ ]:
def generate_response(model, tokeniser, prompt, max_new_tokens=200):
    messages = [
        {"role": "system", "content": "You are a financial analyst assistant."},
        {"role": "user", "content": prompt}
    ]
    formatted = tokeniser.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokeniser(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.3,
            do_sample=True, pad_token_id=tokeniser.eos_token_id
        )
    return tokeniser.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


test_prompt = "What are the most important things to look at when analysing a company's annual report?"

print(f"Prompt: {test_prompt}")
print("=" * 70)

print("\n[Aligned model response]:")
print("-" * 40)
print(generate_response(dpo_trainer.model, dpo_tokeniser, test_prompt))

---

## Summary

In this notebook we:
- Understood the problem alignment solves — teaching models to produce *preferred* outputs, not just correct ones
- Built a preference dataset with chosen/rejected response pairs
- Trained a reward model that scores responses as a scalar value
- Applied DPO — a simpler alternative to full RLHF that achieves similar alignment
- Understood the `beta` hyperparameter: the balance between following preferences and staying close to the original model

We now have a model that is (1) domain-adapted via fine-tuning and (2) preference-aligned via DPO. The final step: measure how much this actually improved.

---

▶ **Next: [Notebook 06 — Evaluation & Benchmarking](06_evaluation.ipynb)**